In [1]:
from functools import reduce

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [2]:
# with open(r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\correct.bin', 'rb') as f:
#     ans_df = pickle.load(file=f)
#     ans_df = ans_df[ans_df['task'] != 'SIQA']

# ans_df.head()

In [3]:
# preds_path_ls = [
#     ('Raw',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\Raw-CR.bin'),
#     ('LoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA-cfg18-CR.bin'),
#     ('LoRA 1b',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA_1b-cfg1-CR.bin'),
#     ('DoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\DoRA-cfg18-CR.bin'),
#     ('VeRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\VeRA-cfg1-CR.bin'),
#     ('GSOFT',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\GSOFT-cfg3-CR.bin'),
# ]

preds_path_ls = [
    ('Raw_1b',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\Raw_1b_CR.bin'),
    ('LoRA_1b-CR',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA_1b-cfg1-CR_CR.bin'),
    ('LoRA_1b-code st-2000',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA_1b-cfg1-st2000_CR.bin'),
    ('LoRA_1b-code st-4000',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA_1b-cfg1-st4000_CR.bin'),
    ('LoRA_1b-code st-6000',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA_1b-cfg1-st6000_CR.bin'),
]

In [4]:
preds_dfs = []

for run_name, preds_path in preds_path_ls:
    with open(preds_path, 'rb') as f:
        preds_df: pd.DataFrame = pickle.load(file=f)

        # preds_df = preds_df[preds_df['task'] != 'SIQA']
        preds_df.insert(loc=len(preds_df.columns), column='run_name', value=run_name)

        # if run_name == 'GSOFT':
        #     preds_df = pd.merge(
        #         left=ans_df,
        #         right=preds_df,
        #         how='inner',
        #         on=['text', 'text_wa_answer', 'task'],
        #         validate='1:1'
        #     )

        preds_dfs.append(preds_df)

In [5]:
preds_df = pd.concat(preds_dfs)

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,{'generated_text': ': False'},Raw_1b
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': False'},Raw_1b
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': False'},Raw_1b
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': TRUE'},Raw_1b
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': True'},Raw_1b


In [6]:
preds_df[['task', 'correct_answer']].value_counts().sort_index()

task        correct_answer
ARC-C       1                    10
            2                     5
            4                     5
            A                   310
            B                   360
            C                   390
            D                   410
            E                     5
ARC-E       1                    30
            2                    35
            3                    25
            4                    15
            A                   760
            B                   725
            C                   685
            D                   575
BoolQ       False              6185
            True              10165
OBQA        A                   630
            B                   690
            C                   550
            D                   630
PIQA        Solution1          4550
            Solution2          4640
SIQA        A                  3215
            B                  3270
            C                  3285
h

In [7]:
sorted(list(preds_df['correct_answer'].unique()))

['1',
 '2',
 '3',
 '4',
 'A',
 'B',
 'C',
 'D',
 'E',
 'Ending0',
 'Ending1',
 'Ending2',
 'Ending3',
 'False',
 'Option1',
 'Option2',
 'Solution1',
 'Solution2',
 'True']

In [8]:
preds_df['run_name'].value_counts()

run_name
Raw_1b                  19740
LoRA_1b-CR              19740
LoRA_1b-code st-2000    19740
LoRA_1b-code st-4000    19740
LoRA_1b-code st-6000    19740
Name: count, dtype: int64

In [9]:
tasks = preds_df.task.unique()

tasks

array(['BoolQ', 'PIQA', 'SIQA', 'hellaswag', 'winogrande', 'ARC-E',
       'ARC-C', 'OBQA'], dtype=object)

In [10]:
import string
import re

punct_trans = str.maketrans(dict.fromkeys(string.punctuation))

def BoolQ_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def SIQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

PIQA_pattern = re.compile('Solution[1,2]')
def PIQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip() if pred else ''
    pred = re.match(PIQA_pattern, pred)
    pred = pred.group(0) if pred else 'None'

    return pred

hellaswag_pattern = re.compile('(Ending)?[0,1,2,3]')
def hellaswag_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(hellaswag_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Ending{pred[-1]}'
    else:
        pred = pred_0

    return pred

winogrande_pattern = re.compile('Option[1,2]')
def winogrande_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(winogrande_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Option{pred[-1]}'
    else:
        pred = pred_0

    return pred

def ARC_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def OBQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

In [11]:
preds_dfs[0][preds_dfs[0]['task'] == 'SIQA']['model_pred'].apply(SIQA_process)

5108    B
5109    A
5110    A
5111    A
5112    A
       ..
7057    C
7058    B
7059    C
7060    B
7061    B
Name: model_pred, Length: 1954, dtype: object

In [12]:
preds_dfs[0][preds_dfs[0]['task'] == 'SIQA']['correct_answer'][5108]

'C'

In [13]:
task_postprocessors = {
    'BoolQ': BoolQ_process,
    'PIQA': PIQA_process,
    'SIQA': SIQA_process,
    'hellaswag': hellaswag_process,
    'winogrande': winogrande_process,
    'ARC-E': ARC_process,
    'ARC-C': ARC_process,
    'OBQA': OBQA_process
}

for task, processor in task_postprocessors.items():
    preds_df.loc[preds_df['task'] == task, 'model_pred'] = preds_df.loc[preds_df['task'] == task, 'model_pred'].apply(processor)

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False,Raw_1b
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw_1b
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw_1b
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,TRUE,Raw_1b
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True,Raw_1b


In [14]:
preds_df.loc[:, 'is_correct'] = (preds_df.loc[:, 'correct_answer'] == preds_df.loc[:, 'model_pred'])

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name,is_correct
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False,Raw_1b,True
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw_1b,False
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw_1b,False
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,TRUE,Raw_1b,False
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True,Raw_1b,True


In [15]:
accuracy = preds_df[['run_name', 'task', 'is_correct']].groupby(by=['run_name', 'task'], as_index=False)['is_correct'].mean()

accuracy

,run_name,task,is_correct
0,LoRA_1b-CR,ARC-C,0.585284
1,LoRA_1b-CR,ARC-E,0.735088
2,LoRA_1b-CR,BoolQ,0.648012
3,LoRA_1b-CR,OBQA,0.694000
4,LoRA_1b-CR,PIQA,0.763874
5,LoRA_1b-CR,SIQA,0.708802
6,LoRA_1b-CR,hellaswag,0.811990
7,LoRA_1b-CR,winogrande,0.563536
8,LoRA_1b-code st-2000,ARC-C,0.538462
9,LoRA_1b-code st-2000,ARC-E,0.643860


In [16]:
accuracy[accuracy['run_name'] == 'GSOFT']

,run_name,task,is_correct


In [17]:
accuracy[['run_name', 'is_correct']].groupby(by=['run_name'], as_index=False).mean()

,run_name,is_correct
0,LoRA_1b-CR,0.688823
1,LoRA_1b-code st-2000,0.594282
2,LoRA_1b-code st-4000,0.522557
3,LoRA_1b-code st-6000,0.507180
4,Raw_1b,0.460116
